- **OPTIMIZE** can be applied at partition level.
- Choosing Partition Columns: 
    - Low Cardinality Columns
    - Partitions should be atleast 1GB in size or larger depending on total table size.
    - Datetime column can be used for partitioning
- Keep in mind:
    - Files cannot be compacted or combined across partition boundaries: Partitioned small tables increase storage costs and total number of files to scan.
    - If most partitions < 1 gb of data, the table is over-partitioned
    - Data that is over partitioned or incorrectly partitioned will suffer greatly
- Deletes also can be done in partitioned tables for removing old data in partitions.
- If that table is then used for stream read it should be read as :
    - spark.readStream.option("**IgnoreDeletes**",True).table(myTable)
    - bcos delete breaks the append only feature of streaming table so it is imp to tell to ignoreDeletes for streaming read.
    - and of course deletion of files will not actually occur until you run VACUUM on table.

Optimizing Techniques :
- Partitioning
- Z-Order Indexing
- Liquid Clustering

- Partitioning can improve performance but only for huge delta tables
- It often results in small files problem
- Repartitioning rquires full table re-write.

ZOrder Indexing :
- Groups similar data into optimized files without creating directories.
- OPTIMIZE mytable **ZORDER BY** column_name
- Leverages data-skipping algorithm
- Effective for high cardinality columns.
- Main Issue : it is not an incremental operation
    - When new data is ingested you must rerun the OPTIMIZE(OPTIMIZE mytable **ZORDER BY** column_name) command to reorganize the data.
    - Re-running the command may result in the recreation of all files in the entire tabl, which can be computationally intensive

Liquid Clustering:
- Improved version of Z-Order indexing with more flexibility and better performance.
- Table level definition, this eliminates need to specify them at each run of the OPTIMIZE command.
- New or existing table using : CLUSTER BY
- Clustering is not compatible with ZOrder or Partitioning
- You can create a new table by projecting the existing table and using the previous partition columns and ZOrder columns as CLUSTERING KEYS.
- To trigger clustering simply use the OPTIMIZE command on your table. 
- This is an incremental operation.
- On new data ingestion it is smart enough to know that old data is already well clustered.
- So on next run of OPTIMIZE command only unoptimized files are affected.
- Already optimized files are simply ignored.
- Clustering keys can be evolved as needed.

Automatic Liquid Clustering:
- Databricks automatically chooses clustering keys by analyzing the table historical query workload.
- **Requires Predictive Optimization on Unity Catalog managed tables.**
- CREATE TABLE table1(col1 int, col2 string) CLUSTER BY AUTO
- ALTER TABLE table2 CLUSTER BY AUTO

Predictive Optimization:
- 